In [3]:
# Sam Brown
# sam_brown@mines.edu
# June 11 2025
# Goal: Use the Day class to parse our data and store high and low tide events for each day, eventually creating a feature that identifies which kind of event it is and if the low tide event was skipped.

import my_lib.funcs
import Day

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# We will use 2011 to 2013 first to make sure code works, then expand to larger time scales to see if we can find patterns
events_list2011 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2011_2011Events2stas")
events_list2012 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2012_2012Events2stas")
events_list2013 = my_lib.funcs.load_evt("/Users/sambrown04/Documents/SURF/Events/2013_2013Events2stas")


# Use preprocessing function to get some of the features
pre_11 = my_lib.funcs.extract_event_features(events_list2011)
pre_12 = my_lib.funcs.extract_event_features(events_list2012)
pre_13 = my_lib.funcs.extract_event_features(events_list2013)

# Merge into one large list of Dataframes
tot_dat = pre_11 + pre_12 + pre_13

# List to store instances of Day class
days = []

In [7]:
tot_dat[0].dtypes

station                  object
pre-slip_area           float64
slip_severity           float64
peak_time               float64
total_delta             float64
start_time       datetime64[ns]
dtype: object

In [9]:
# We will use the average coordinates for gz stations to retrieve our tide data
x_cor = -168955.1491394913 
y_cor = -599694.5432784811

tide_df = my_lib.funcs.get_tide_height(1100, x_cor, y_cor, "2011-01-01 00:00:00") # tide height is in centimeters (1100 days = 3 years)

Elapsed time: 30.460657835006714 seconds


In [10]:
# Only down to minutes, need to double check, some values are getting missed
tide_df['time'] = tide_df['time'].apply(lambda x: x.strftime("%Y-%m-%d %H:%M"))

for df in tot_dat:
    df['start_time'] = df['start_time'].apply(lambda x: x.strftime("%Y-%m-%d %H:%M"))

tot_dat = sorted(tot_dat, key=lambda df: df['start_time'].iloc[0])   

In [26]:
class Event:
    def __init__(self, high, date, tide_h):
        self.high = high
        self.date = date
        self.tide_h = tide_h
    

In [27]:
Events = []

for event in tot_dat:
    date_time = event.at[0, 'start_time']
    # print(date_time)

    index = tide_df[tide_df['time'] == date_time].index
    
    tide = tide_df.at[index[0], 'tide_height']
    if tide > 0:
        tide = True

    else:
        tide = False

    curEvent = Event(tide, date_time, tide)
    Events.append(curEvent)
        
    

    